In [1]:
import pandas as pd
import polars as pl

In [2]:
def dms_to_decimal(dms):
    """Convert DMS packed as ±DDMMSS or ±DDDMMSS to decimal degrees."""
    sign = -1 if str(dms).startswith("-") else 1
    dms = abs(int(dms))

    degrees = dms // 10000
    minutes = (dms % 10000) // 100
    seconds = dms % 100

    val = round(sign * (degrees + minutes / 60 + seconds / 3600), 4)
    return val

In [3]:
# Add decimal lat, lon and plot_id

df_plm = pl.read_csv("../data/raw/ICP/595_mm_20260227091917/mm_plm.csv", separator=";")

df_plm = df_plm.with_columns(
    [
        pl.col("latitude").map_elements(dms_to_decimal, return_dtype=pl.Float64).alias("Lat"),
        pl.col("longitude").map_elements(dms_to_decimal, return_dtype=pl.Float64).alias("Lon"),
    ]
)

df_plm = df_plm.with_columns(
    (
        pl.col("code_country").cast(pl.Utf8).str.zfill(2)
        + "."
        + pl.col("code_plot").cast(pl.Utf8).str.zfill(4)
    ).alias("plot_id")
)

df_plm.head()

survey_year,code_country,partner_code,code_plot,instrument_seq_nr,code_location,latitude,longitude,code_altitude,code_variable,position_vertical,code_recording,scanning_intervall,storing_intervall,sw_id,date_monitoring_first,date_monitoring_last,days_measuring,instrument_desc,other_obs,q_flag,change_date,code_line,line_nr,Lat,Lon,plot_id
i64,i64,i64,i64,i64,str,str,str,i64,str,f64,i64,f64,f64,str,str,str,i64,str,str,str,str,str,i64,f64,f64,str
1994,1,1,6,2,"""F""","""+501200""","""+034300""",4,"""AT""",1.5,50,60.0,30.0,null,"""1994-12-10""","""1994-12-31""",22,"""CHP 59""","""code_plot_instr: 6.02""",null,"""2009-12-06 22:40:00""","""MMFR1994-002378""",2378,50.2,3.7167,"""01.0006"""
1994,1,1,6,1,"""F""","""+501200""","""+034300""",4,"""PR""",1.0,50,60.0,30.0,null,"""1994-12-10""","""1994-12-31""",22,"""CHP 59""","""code_plot_instr: 6.01""",null,"""2009-12-06 22:40:00""","""MMFR1994-002189""",2189,50.2,3.7167,"""01.0006"""
1994,1,1,6,3,"""F""","""+501200""","""+034300""",4,"""RH""",1.5,50,60.0,30.0,null,"""1994-12-10""","""1994-12-31""",22,"""CHP 59""","""code_plot_instr: 6.03""",null,"""2009-12-06 22:40:00""","""MMFR1994-002188""",2188,50.2,3.7167,"""01.0006"""
1994,1,1,16,1,"""F""","""+481100""","""-013400""",2,"""PR""",1.0,50,60.0,30.0,null,"""1994-12-07""","""1994-12-31""",25,"""CHS 35""","""code_plot_instr: 16.01""",null,"""2009-12-06 22:40:00""","""MMFR1994-002379""",2379,48.1833,-1.5667,"""01.0016"""
1994,1,1,16,3,"""F""","""+481100""","""-013400""",2,"""RH""",1.5,50,60.0,30.0,null,"""1994-12-07""","""1994-12-31""",25,"""CHS 35""","""code_plot_instr: 16.03""",null,"""2009-12-06 22:40:00""","""MMFR1994-002380""",2380,48.1833,-1.5667,"""01.0016"""


In [4]:
# -------------------------------------------------------------------
# Data loading and preprocessing for ICP MM data
#
# This cell performs the following steps:
#
# 1. Reads the raw ICP MM CSV file (semicolon-separated).
# 2. Perform a snaity check to ensure that mean value is between min and max.
# 3. Generates a unique `plot_id` by zero-padding and combining
#    `code_country` and `code_plot` into the format CC.PPPP.
# 4. Converts the `date_observation` column from string to `pl.Date`.
# 5. Extracts `year` and `month` from the observation date to support
#    temporal filtering and aggregation.
# 6. Creates a `month_year` column in MM-YYYY format for monthly aggregation.
# 7. Filters out historical observations from 1960 and earlier, which are
#    outside the scope of the analysis.
#
# The resulting DataFrame is prepared for downstream grouping,
# aggregation, and pivot operations.
# -------------------------------------------------------------------

df = pl.read_csv("../data/raw/ICP/595_mm_20260227091917/mm_mem.csv", separator=";")

df = df.with_columns(
    pl.col("daily_min").cast(pl.Float64),
    pl.col("daily_mean").cast(pl.Float64),
    pl.col("daily_max").cast(pl.Float64),
)

df = df.with_columns(
    [
        pl.when(pl.col("daily_mean") < pl.col("daily_min"))
        .then(None)
        .otherwise(pl.col("daily_min"))
        .alias("daily_min"),
        pl.when(pl.col("daily_mean") > pl.col("daily_max"))
        .then(None)
        .otherwise(pl.col("daily_max"))
        .alias("daily_max"),
    ]
)

df = (
    df.with_columns(
        (
            pl.col("code_country").cast(pl.Utf8).str.zfill(2)
            + "."
            + pl.col("code_plot").cast(pl.Utf8).str.zfill(4)
        ).alias("plot_id")
    )
    .with_columns(
        pl.col("date_observation").str.strptime(pl.Date, "%Y-%m-%d").alias("date_observation")
    )
    .with_columns(
        [
            pl.col("date_observation").dt.year().alias("year"),
            pl.col("date_observation").dt.month().alias("month"),
        ]
    )
    .with_columns(pl.col("date_observation").dt.strftime("%m-%Y").alias("month_year"))
    .filter(pl.col("year") > 1960)
)

df.head()

survey_year,code_country,partner_code,code_plot,instrument_seq_nr,code_variable,date_observation,daily_mean,daily_min,daily_max,daily_completeness,code_data_origin,code_data_status,other_obs,q_flag,change_date,code_line,line_nr,plot_id,year,month,month_year
i64,i64,i64,i64,i64,str,date,f64,f64,f64,i64,i64,i64,str,str,str,str,i64,str,i32,i8,str
1994,1,1,6,3,"""RH""",1994-12-30,79.7,69.0,91.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112559""",112559,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-31,90.1,84.0,94.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112562""",112562,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-14,89.7,59.0,100.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112610""",112610,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-15,92.8,73.0,98.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112612""",112612,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-16,94.8,84.0,100.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112615""",112615,"""01.0006""",1994,12,"""12-1994"""


In [5]:
df_ch = df.filter(pl.col("code_country")==50)

df_ch.select("code_plot").unique().to_series().to_list()

[9, 6, 3, 18, 15, 12, 4, 7, 10, 13, 1, 16, 19, 5, 14, 11, 2, 8]

In [6]:
# There are duplicate observations for the same plot, variable, and date
# caused by multiple records with different `code_line` and `line_nr` values.
# These duplicates represent the same measurement context and should be
# consolidated into a single record.
#
# To resolve this, we group by the unique identifiers that define a single
# observation (country, plot, variable, date, plot_id, and month_year),
# and compute the mean of all remaining numeric columns across duplicates.
# This effectively averages over multiple entries for the same observation.
#
# After aggregation, we drop metadata and quality-control columns that are
# no longer meaningful once the duplicates have been collapsed.

df = (
    df.group_by(
        ["code_country", "code_plot", "code_variable", "date_observation", "plot_id", "month_year"]
    )
    .agg(
        [
            pl.all()
            .exclude(
                [
                    "code_country",
                    "code_plot",
                    "code_variable",
                    "date_observation",
                    "plot_id",
                    "month_year",
                ]
            )
            .mean()
        ]
    )
    .drop(
        [
            "code_data_origin",
            "code_data_status",
            "other_obs",
            "q_flag",
            "change_date",
            "code_line",
            "line_nr",
        ]
    )
)

df.head()

code_country,code_plot,code_variable,date_observation,plot_id,month_year,survey_year,partner_code,instrument_seq_nr,daily_mean,daily_min,daily_max,daily_completeness,year,month
i64,i64,str,date,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,21,"""PR""",2018-06-18,"""02.0021""","""06-2018""",2018.0,102.0,49.0,0.0,0.0,0.0,100.0,2018.0,6.0
15,18,"""RH""",2004-10-26,"""15.0018""","""10-2004""",2004.0,15.0,59.0,97.6,93.4,100.0,100.0,2004.0,10.0
54,206,"""AT""",2013-06-19,"""54.0206""","""06-2013""",2013.0,54.0,1.0,23.9,18.2,30.2,100.0,2013.0,6.0
58,2061,"""SR""",2017-01-29,"""58.2061""","""01-2017""",2017.0,58.0,1.0,54.8,null,null,100.0,2017.0,1.0
58,2401,"""WC""",2013-12-08,"""58.2401""","""12-2013""",2013.0,58.0,5.0,32.744444,32.7,32.777778,100.0,2013.0,12.0


In [7]:
df_pivoted = df.pivot(
    values=["daily_min", "daily_max", "daily_mean", "daily_completeness"],
    index=[
        "code_country",
        "code_plot",
        "plot_id",
        "month_year",
        "date_observation",
    ],
    on="code_variable",
)
df_pivoted = df_pivoted.select(
    [
        "code_country",
        "code_plot",
        "plot_id",
        "date_observation",
        "month_year",
        "daily_mean_PR",
        "daily_completeness_PR",
        "daily_mean_AT",
        "daily_min_AT",
        "daily_max_AT",
        "daily_completeness_AT",
        "daily_mean_RH",
        "daily_min_RH",
        "daily_max_RH",
        "daily_completeness_RH",
        "daily_mean_WS",
        "daily_max_WS",
        "daily_completeness_WS",
        "daily_mean_WD",
        "daily_completeness_WD",
        "daily_mean_SR",
        "daily_completeness_SR",
    ]
)
df_pivoted.head()

code_country,code_plot,plot_id,date_observation,month_year,daily_mean_PR,daily_completeness_PR,daily_mean_AT,daily_min_AT,daily_max_AT,daily_completeness_AT,daily_mean_RH,daily_min_RH,daily_max_RH,daily_completeness_RH,daily_mean_WS,daily_max_WS,daily_completeness_WS,daily_mean_WD,daily_completeness_WD,daily_mean_SR,daily_completeness_SR
i64,i64,str,date,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,21,"""02.0021""",2018-06-18,"""06-2018""",0.0,100.0,16.748801,14.20001,20.00999,100.0,74.908403,61.295,86.615,100.0,1.228333,2.638334,100.0,227.6776,100.0,116.306798,100.0
15,18,"""15.0018""",2004-10-26,"""10-2004""",4.6,100.0,8.3,5.65,10.15,100.0,97.6,93.4,100.0,100.0,4.4,10.7,100.0,187.3,100.0,1.9,100.0
54,206,"""54.0206""",2013-06-19,"""06-2013""",null,null,23.9,18.2,30.2,100.0,75.5,52.9,99.9,100.0,2.6,9.1,100.0,45.0,100.0,288.5,100.0
58,2061,"""58.2061""",2017-01-29,"""01-2017""",0.0,100.0,-8.7,-11.2,-4.3,100.0,99.6,94.9,100.0,100.0,0.3,1.0,100.0,180.0,100.0,54.8,100.0
58,2401,"""58.2401""",2013-12-08,"""12-2013""",0.0,100.0,-2.5,-4.6,0.8,100.0,97.2,83.0,100.0,100.0,1.2,2.7,100.0,135.0,100.0,14.2,100.0


In [8]:
# -------------------------------------------------------------------
# Variable-wise data completeness analysis
#
# This function analyzes data availability for a given variable
# (e.g., air temperature "AT") across plots and months.
#
# Execution steps:
# 1. Filter the input DataFrame to keep only rows corresponding to the
#    specified `code_variable`.
# 2. Report the number of unique month–year combinations available
#    for the selected variable, providing a quick overview of its
#    temporal coverage.
# 3. Group the filtered data by country, plot, and month–year.
# 4. For each group, check whether all values of `daily_mean`,
#    `daily_min`, and `daily_max` are missing (null).
# 5. Identify months where *all three* daily statistics are completely
#    null, indicating a full absence of usable data for that variable
#    in that plot and month.
# 6. Sort the resulting records chronologically by month–year.
#
# The function returns a DataFrame listing plot–month combinations
# where the selected variable has no valid observations at all.
# -------------------------------------------------------------------


def variable_wise_analysis(df: pl.DataFrame, variable: str):
    """Variable wise analysis for missing data."""
    tdf = df.filter(pl.col("code_variable") == variable)
    print(
        "Number of unique (month-year) data in ",
        variable,
        "is :",
        tdf.select(pl.col("month_year")).n_unique(),
    )
    null_months = (
        tdf.group_by(["code_country", "code_plot", "month_year"])
        .agg(
            [
                (pl.col("daily_mean").is_null().all()).alias("daily_mean_all_null"),
                (pl.col("daily_min").is_null().all()).alias("daily_min_all_null"),
                (pl.col("daily_max").is_null().all()).alias("daily_max_all_null"),
            ]
        )
        .filter(
            pl.col("daily_mean_all_null")
            & pl.col("daily_min_all_null")
            & pl.col("daily_max_all_null")
        )
    ).sort(by="month_year")
    return null_months


null_months = variable_wise_analysis(df, "AT")

null_months.head()

Number of unique (month-year) data in  AT is : 384


code_country,code_plot,month_year,daily_mean_all_null,daily_min_all_null,daily_max_all_null
i64,i64,str,bool,bool,bool
4,308,"""01-1994""",true,true,true
4,308,"""01-1996""",true,true,true
2,16,"""01-1996""",true,true,true
2,16,"""01-1997""",true,true,true
4,303,"""01-1997""",true,true,true


In [9]:
# -------------------------------------------------------------------
# Monthly aggregation of daily climate variables
#
# This block computes monthly summary statistics for each plot and
# variable based on daily observations.
#
# Execution steps:
# 1. Replace NaN values in daily statistics with nulls to ensure
#    correct aggregation behavior in Polars.
# 2. Group data by plot, variable, and calendar month
#    (`plot_id`, `code_variable`, `month_year`, `year`, `month`).
# 3. Compute monthly aggregates:
#    - `avg_daily_mean`: mean of daily mean values within the month
#    - `min_daily_min`: minimum of daily minimum values (monthly extreme)
#    - `max_daily_max`: maximum of daily maximum values (monthly extreme)
#    - `frost_days`: count of days with minimum temperature below 0°C
# 4. Sort the resulting monthly summaries chronologically by year
#    and month.
#
# The resulting DataFrame provides a compact monthly representation
# of daily climate dynamics for each plot and variable.
# -------------------------------------------------------------------

monthly_summary = (
    df.with_columns(
        [
            pl.col("daily_mean").fill_nan(None),
            pl.col("daily_min").fill_nan(None),
            pl.col("daily_max").fill_nan(None),
        ]
    )
    .group_by(
        [
            "plot_id",
            "code_country",
            "code_plot",
            "code_variable",
            "month_year",
            "year",
            "month",
        ]
    )
    .agg(
        [
            pl.col("daily_mean").mean().alias("avg_daily_mean"),
            pl.col("daily_min").min().alias("min_daily_min"),
            pl.col("daily_max").max().alias("max_daily_max"),
            (pl.col("daily_min") < 0).sum().alias("frost_days"),
            pl.col("daily_completeness").mean().alias("avg_completeness"),
        ]
    )
    .sort(["year", "month"])
)

monthly_summary.head()

plot_id,code_country,code_plot,code_variable,month_year,year,month,avg_daily_mean,min_daily_min,max_daily_max,frost_days,avg_completeness
str,i64,i64,str,str,f64,f64,f64,f64,f64,u32,f64
"""07.0001""",7,1,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0011""",7,11,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0109""",7,109,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0010""",7,10,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0010""",7,10,"""PR""","""02-1991""",1991.0,2.0,0.0,null,null,0,0.0


In [10]:
monthly_summary.filter((pl.col("code_variable") == "SR") & (pl.col("plot_id") == "02.0008"))

plot_id,code_country,code_plot,code_variable,month_year,year,month,avg_daily_mean,min_daily_min,max_daily_max,frost_days,avg_completeness
str,i64,i64,str,str,f64,f64,f64,f64,f64,u32,f64
"""02.0008""",2,8,"""SR""","""01-1996""",1996.0,1.0,628.967742,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""02-1996""",1996.0,2.0,792.862069,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""03-1996""",1996.0,3.0,1891.354839,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""04-1996""",1996.0,4.0,3347.233333,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""05-1996""",1996.0,5.0,2835.935484,null,null,0,100.0
…,…,…,…,…,…,…,…,…,…,…,…
"""02.0008""",2,8,"""SR""","""08-2025""",2025.0,8.0,213.845161,0.0,841.0,0,95.16129
"""02.0008""",2,8,"""SR""","""09-2025""",2025.0,9.0,136.173333,0.0,740.7,0,94.866667
"""02.0008""",2,8,"""SR""","""10-2025""",2025.0,10.0,72.096774,0.0,614.5,0,95.032258


In [11]:
plot_temporal_coverage = (
    monthly_summary.with_columns(
        # Ensure month_year is a proper date
        pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")
    )
    .group_by("plot_id", "code_variable")
    .agg(
        [
            pl.len().alias("n_rows"),  # number of rows
            pl.col("month_year").min().alias("min_month_year"),  # earliest month
            pl.col("month_year").max().alias("max_month_year"),  # latest month
            (
                (pl.col("month_year").max().dt.year() - pl.col("month_year").min().dt.year()) * 12
                + (pl.col("month_year").max().dt.month() - pl.col("month_year").min().dt.month())
                + 1
            ).alias("n_months"),  # months span
        ]
    )
    .sort("plot_id")
)

plot_temporal_coverage.head()

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months
str,str,u32,date,date,i32
"""01.0003""","""PR""",146,1995-06-01,2008-07-01,158
"""01.0003""","""AT""",146,1995-06-01,2008-07-01,158
"""01.0003""","""RH""",146,1995-06-01,2008-07-01,158
"""01.0006""","""AT""",348,1994-12-01,2024-12-01,361
"""01.0006""","""PR""",348,1994-12-01,2024-12-01,361


In [12]:
def missing_months(min_date, max_date, existing_months):
    """Create all months between min and max months."""
    all_months = pd.date_range(start=min_date, end=max_date, freq="MS").strftime("%m-%Y").to_list()

    missing = [d for d in all_months if d not in existing_months]
    return missing


incomplete_plots = plot_temporal_coverage.filter(pl.col("n_rows") != pl.col("n_months"))

# Collect month lists for each plot
missing_months_list = []

for row in plot_temporal_coverage.iter_rows(named=True):
    plot_id = row["plot_id"]
    code_var = row["code_variable"]
    min_month = row["min_month_year"]
    max_month = row["max_month_year"]

    # Existing months for this plot
    existing_months = (
        monthly_summary.filter((pl.col("plot_id") == plot_id) & (pl.col("code_variable")==code_var))
        .select("month_year")
        .to_series()
        .to_list()
    )

    missing = missing_months(min_month, max_month, existing_months)

    missing_months_list.append(
        {"plot_id": plot_id, "code_variable":code_var, "n_missing_months": len(missing), "missing_months": missing}
    )

# Convert to DataFrame
missing_months_df = pl.DataFrame(missing_months_list)

plot_temporal_coverage = plot_temporal_coverage.join(missing_months_df, on=["plot_id", "code_variable"])

plot_temporal_coverage.head()

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months
str,str,u32,date,date,i32,i64,list[str]
"""01.0003""","""PR""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""AT""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""RH""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0006""","""AT""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]"
"""01.0006""","""PR""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]"


In [13]:
plot_temporal_coverage = plot_temporal_coverage.join(missing_months_df, on=["plot_id", "code_variable"])

plot_temporal_coverage.head()

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months,n_missing_months_right,missing_months_right
str,str,u32,date,date,i32,i64,list[str],i64,list[str]
"""01.0003""","""PR""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]",12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""AT""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]",12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""RH""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]",12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0006""","""AT""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]",13,"[""01-2001"", ""02-2001"", … ""12-2021""]"
"""01.0006""","""PR""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]",13,"[""01-2001"", ""02-2001"", … ""12-2021""]"


In [14]:
plot_temporal_coverage.filter(pl.col("n_missing_months") == 0)

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months,n_missing_months_right,missing_months_right
str,str,u32,date,date,i32,i64,list[str],i64,list[str]
"""02.0001""","""ST""",328,1998-09-01,2025-12-01,328,0,[],0,[]
"""02.0001""","""RH""",328,1998-09-01,2025-12-01,328,0,[],0,[]
"""02.0001""","""WD""",36,2001-01-01,2003-12-01,36,0,[],0,[]
"""02.0001""","""WS""",328,1998-09-01,2025-12-01,328,0,[],0,[]
"""02.0001""","""AT""",328,1998-09-01,2025-12-01,328,0,[],0,[]
…,…,…,…,…,…,…,…,…,…
"""67.0005""","""SR""",144,2013-01-01,2024-12-01,144,0,[],0,[]
"""67.0005""","""AT""",144,2013-01-01,2024-12-01,144,0,[],0,[]
"""67.0005""","""WS""",144,2013-01-01,2024-12-01,144,0,[],0,[]


In [15]:
# plot_id = "02.0008" has 360 months of consecutive data,
# so we select this plot for sample simulations

plot_df = df.filter(pl.col("plot_id") == "02.0008")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").min().alias("tmp_min"),
    pl.col("daily_max").max().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
    .to_pandas()
)

# weather_df.to_excel("../data/intermediate/weather_data.xlsx", index=False)

weather_df

,year,month,tmp_ave,tmp_min,tmp_max,frost_days,prcp,srad
0,1996,1,3.067742,-5.50,13.85,12,20.0,628.967742
1,1996,2,2.032759,-6.70,10.30,14,110.5,792.862069
2,1996,3,3.490323,-7.05,18.30,19,23.6,1891.354839
3,1996,4,9.005000,-4.65,27.50,10,6.9,3347.233333
4,1996,5,10.140323,-3.40,28.20,3,79.2,2835.935484
...,...,...,...,...,...,...,...,...
355,2025,8,17.617742,5.55,33.35,0,26.3,213.845161
356,2025,9,13.885000,4.35,27.45,0,73.8,136.173333
357,2025,10,10.619355,2.35,18.25,0,91.1,72.096774
358,2025,11,6.713333,-7.10,16.20,8,79.5,36.536667


In [16]:
plot_temporal_coverage.filter(pl.col("plot_id")=="50.0018")

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months,n_missing_months_right,missing_months_right
str,str,u32,date,date,i32,i64,list[str],i64,list[str]
"""50.0018""","""WD""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""WS""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""AT""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""PR""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""RH""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""SR""",121,2013-01-01,2023-01-01,121,0,[],0,[]


In [ ]:
# Plot: Davos

plot_df = df.filter(pl.col("plot_id") == "50.0018")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").min().alias("tmp_min"),
    pl.col("daily_max").max().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
    .to_pandas()
)

weather_df.to_excel("../data/intermediate/Davos_weather_data.xlsx", index=False)

weather_df

,year,month,tmp_ave,tmp_min,tmp_max,frost_days,prcp,srad
0,2013,1,-4.078065,-17.80,7.90,31,22.0,59.883871
1,2013,2,-7.617500,-18.40,5.50,28,30.3,105.521429
2,2013,3,-1.813871,-16.60,9.50,30,19.3,143.580645
3,2013,4,3.561000,-12.00,15.60,16,24.4,201.916667
4,2013,5,4.560323,-3.80,15.50,11,89.7,179.774194
...,...,...,...,...,...,...,...,...
116,2022,9,7.769000,-1.39,22.15,3,77.5,148.756000
117,2022,10,9.123871,1.09,19.51,0,75.3,113.648065
118,2022,11,1.748000,-6.48,13.52,19,37.6,67.776000
119,2022,12,-1.801333,-15.50,8.79,26,30.0,49.346667


In [24]:


plot_df = df.filter(pl.col("plot_id") == "50.0013")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").min().alias("tmp_min"),
    pl.col("daily_max").max().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
    .to_pandas()
)

# weather_df.to_excel("../data/intermediate/weather_data.xlsx", index=False)

weather_df

,year,month,tmp_ave,tmp_min,tmp_max,frost_days,prcp,srad
0,1999,1,1.488710,-7.25,11.750,18,26.750,27.803226
1,1999,2,-0.416071,-10.20,10.250,20,58.900,25.532143
2,1999,3,6.216129,-1.45,18.600,3,25.950,87.087097
3,1999,4,9.016667,0.00,21.400,0,41.550,127.889286
4,1999,5,15.075806,7.50,29.000,0,112.750,158.025806
...,...,...,...,...,...,...,...,...
295,2023,8,19.577581,9.34,35.305,0,77.205,155.357419
296,2023,9,18.142333,7.13,31.310,0,71.670,142.337333
297,2023,10,12.568548,2.72,25.300,0,61.730,58.848387
298,2023,11,5.368000,-1.58,13.525,7,182.110,19.072333
